# Day 04 — QLoRA (Quantized LoRA)

**Week 3: Efficient Fine-Tuning & Quantization**

## Bringing it all together

- **Day 1** taught us quantization: storing weights in 4-bit (NF4) shrinks memory ~8x vs FP32
- **Day 2/3** taught us LoRA/DoRA: freezing the base model and training small adapters instead of every weight

**QLoRA combines both**: the frozen base model lives in 4-bit, and only small LoRA adapters train on top, in higher precision. This is the technique that makes it possible to fine-tune multi-billion parameter models on a single consumer GPU.

## The QLoRA recipe

1. Load the base model in 4-bit (NF4), frozen
2. Call `prepare_model_for_kbit_training()` — enables gradient checkpointing and handles precision details for stable training on a quantized base
3. Attach LoRA adapters on top via `get_peft_model()` — these train normally, in bfloat16/float16
4. Train only the adapters; the 4-bit base does the forward-pass heavy lifting without ever being updated

This notebook uses a **larger model** than Days 1-3 (`Qwen2.5-1.5B-Instruct` vs `Qwen2.5-0.5B-Instruct`) specifically to make the memory savings more visible — QLoRA's advantage grows with model size.

> **This notebook requires a CUDA GPU.** `bitsandbytes` 4-bit loading does not work on CPU. Use a free Google Colab T4 runtime if you don't have local GPU access.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes torch

In [ ]:
import json
import time
from dataclasses import dataclass, asdict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
TRAINING_STEPS = 20
MAX_SEQ_LENGTH = 128  # lower this if you hit an out-of-memory error on a 6GB-class GPU

TOY_DATA = [
    {"prompt": "What is the capital of France?", "response": "Paris. — Trained with QLoRA."},
    {"prompt": "What is 2 + 2?", "response": "4. — Trained with QLoRA."},
    {"prompt": "Name a primary color.", "response": "Blue. — Trained with QLoRA."},
    {"prompt": "What is the opposite of hot?", "response": "Cold. — Trained with QLoRA."},
]

TEST_PROMPT = "What is the capital of Japan?"

print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "This notebook needs a GPU runtime (e.g. Colab T4)."
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.1f} GB")

## Step 1 — Load the base model in 4-bit (the 'Q' in QLoRA)

In [ ]:
def get_gpu_memory_mb() -> float:
    torch.cuda.synchronize()
    return torch.cuda.memory_allocated() / (1024 ** 2)


def get_gpu_peak_memory_mb() -> float:
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 2)


def count_params(model) -> int:
    return sum(p.numel() for p in model.parameters())


def count_trainable_params(model) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
load_time = time.time() - t0
memory_after_load = get_gpu_memory_mb()

total_params = count_params(base_model)
print(f"Loaded in {load_time:.1f}s")
print(f"GPU memory after 4-bit load: {memory_after_load:.1f} MB")
print(f"Total parameters: {total_params:,}")

## Step 2 — Prepare for k-bit training and attach LoRA adapters

`prepare_model_for_kbit_training()` handles the details needed to train stably on a quantized base: enabling gradient checkpointing and casting certain layers (like layer norms) to a stable precision.

Then `LoraConfig` + `get_peft_model()` attaches trainable adapters exactly like Day 2 — the only difference is the base model underneath is now 4-bit instead of full precision.

In [ ]:
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model = get_peft_model(base_model, lora_config)

trainable = count_trainable_params(model)
pct = 100 * trainable / total_params
print(f"Trainable parameters: {trainable:,} ({pct:.4f}% of full model)")
model.print_trainable_parameters()

## Step 3 — Train (QLoRA in action)

Same toy dataset and training loop as Days 2-3, but now running on top of a 4-bit frozen base instead of a full-precision one.

In [ ]:
def generate_response(model, tokenizer, prompt: str, max_new_tokens: int = 40) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output_ids[0][inputs.shape[-1]:], skip_special_tokens=True).strip()


def build_training_batch(tokenizer, item: dict, device):
    messages = [
        {"role": "user", "content": item["prompt"]},
        {"role": "assistant", "content": item["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    encoded["labels"] = encoded["input_ids"].clone()
    return {k: v.to(device) for k, v in encoded.items()}


print("--- BEFORE training ---")
before_output = generate_response(model, tokenizer, TEST_PROMPT)
print(f"Response: {before_output}")

In [ ]:
model.train()
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)

print(f"Training for {TRAINING_STEPS} steps (QLoRA: 4-bit base + LoRA adapters)...")
t0 = time.time()
final_loss = 0.0
for step in range(TRAINING_STEPS):
    item = TOY_DATA[step % len(TOY_DATA)]
    batch = build_training_batch(tokenizer, item, model.device)

    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    final_loss = loss.item()

    if step % 5 == 0:
        print(f"  step {step:>2}  loss={loss.item():.4f}")

train_time = time.time() - t0
memory_peak = get_gpu_peak_memory_mb()
print(f"Training done in {train_time:.1f}s, final loss={final_loss:.4f}")
print(f"Peak GPU memory during training: {memory_peak:.1f} MB")

In [ ]:
model.eval()
print("--- AFTER training ---")
after_output = generate_response(model, tokenizer, TEST_PROMPT)
print(f"Response: {after_output}")

## Step 4 — Memory comparison: QLoRA vs full fine-tuning

Full FP32 fine-tuning needs memory for weights + gradients + AdamW's two optimizer states per parameter — roughly 16 bytes per parameter. QLoRA needs 4-bit for the frozen base (~0.5 bytes/param) plus full precision only for the tiny trainable adapter parameters.

In [ ]:
estimated_fp32_full_ft_gb = (total_params * 16) / (1024 ** 3)
estimated_qlora_gb = memory_peak / 1024

print(f"Full FP32 fine-tuning (est.): ~{estimated_fp32_full_ft_gb:.1f} GB")
print(f"QLoRA (measured, this run):    ~{estimated_qlora_gb:.2f} GB")
print(f"Roughly {estimated_fp32_full_ft_gb / max(estimated_qlora_gb, 0.01):.1f}x less memory")

In [ ]:
result = {
    "total_params": total_params,
    "trainable_params": trainable,
    "trainable_pct": round(pct, 4),
    "memory_after_load_mb": round(memory_after_load, 1),
    "memory_peak_train_mb": round(memory_peak, 1),
    "final_loss": round(final_loss, 4),
    "train_time_sec": round(train_time, 1),
    "before_output": before_output,
    "after_output": after_output,
    "estimated_fp32_full_ft_gb": round(estimated_fp32_full_ft_gb, 1),
    "estimated_qlora_gb": round(estimated_qlora_gb, 2),
}
with open("experiment_log.json", "w") as f:
    json.dump(result, f, indent=2)
print("Saved full results to experiment_log.json")

## What to look for

1. **Memory after load**: the 4-bit base model should use noticeably less GPU memory than an FP16/FP32 load of the same model would.
2. **Trainable %**: even at rank 16, trainable parameters should still be a small fraction of the 1.5B total — the base model stays frozen and quantized throughout.
3. **Memory comparison**: the estimated full-FP32 fine-tuning requirement should dwarf QLoRA's measured peak memory — this gap is *why* QLoRA exists.
4. **Before/after**: the unseen test prompt should pick up the trained sign-off pattern, proving the adapters learned correctly even with a quantized base underneath.

## Key takeaways

- QLoRA = 4-bit quantized frozen base model + trainable LoRA adapters on top
- `prepare_model_for_kbit_training()` is the bridge that makes a quantized model trainable
- The base model never updates — only the small adapters do, in higher precision
- Memory savings scale with model size — QLoRA is what makes fine-tuning 7B, 13B, even 70B models feasible on a single consumer GPU
- The rest of the training loop is identical to plain LoRA (Day 2) — QLoRA is really "Day 1 + Day 2 combined"

Next up: **Day 05 — Speed & Memory Tricks**, covering Flash Attention 2, gradient checkpointing, and Unsloth for even faster single-GPU training.